### Attribution
Sibling notebook to `nps_miprov2.ipynb` in this folder. Both are adapted from:
https://github.com/miptgirl/miptgirl_medium/blob/main/dspy_example/nps_topic_modelling.ipynb

**This variant** replaces `dspy.MIPROv2` with `dspy.GEPA` for the first prompt-optimisation step. The `dspy.BootstrapFewShotWithRandomSearch` section later in the notebook is unchanged, so the two optimisers can be compared side-by-side on the same NPS topic-classification task.

* TODO, create a conventional tran/val/test split

In [1]:
import pandas as pd
import tqdm
from utils import wrap_text

### Try out DSPy on a simple example

In [2]:

SMALL_MODEL_CANDIDATES: list[str] = [
    "gemini/gemini-2.5-flash-lite",
    "gemini/gemini-2.5-flash",
    "gemini/gemini-2.0-flash",
]

REFLECTION_MODEL_CANDIDATES: list[str] = [
    "gemini/gemini-2.5-pro",
    "gemini/gemini-3.1-pro-preview",
    "gemini/gemini-1.5-pro",
]
import dspy
llm = dspy.LM(SMALL_MODEL_CANDIDATES[0])
dspy.configure(lm=llm)
dspy.configure_cache(enable_memory_cache=False, enable_disk_cache=False)

#### Cross-check

# NPS
**Definition**:Net Promoter Score (NPS) is a customer loyalty metric that measures the likelihood of customers recommending a company, ranging from -100 to 100. It is calculated by subtracting the percentage of detractors (0–6 score) from promoters (9–10 score). A score above 0 is good, 50+ is excellent, and 70+ is world-class


In [3]:
import json
with open('nps_comments.json', 'r') as f:
    nps_data = json.loads(f.read())

print(f"sample:\n {nps_data[0]}")

sample:
 {'topics': ['Limited Size or Shade Availability'], 'comment': "Absolutely frustrated! Every time I find something I love, it's sold out in my size. What's the point of having a wishlist if nothing is ever available?"}


#### Get all topics

In [4]:
topics = set()

for rec in nps_data: 
    for t in rec['topics']: 
        topics.add(t)

In [5]:
topic_list = list(topics)
print(wrap_text(str(topic_list)))
print(f"Topic count: {len(topic_list)}")

['Inaccurate Product Descriptions or Photos', 'Limited Size or Shade
Availability', 'Website or App Bugs', 'Unresponsive or Generic Customer
Support', 'Damaged or Incorrect Items', 'Complicated Returns or Exchanges',
'Confusing Loyalty or Discount Systems', 'Customs and Import Charges',
'Difficult Product Discovery', 'Slow or Unreliable Shipping']
Topic count: 10


#### Prompt Optimization with GEPA

GEPA (Genetic-Pareto Evolving Prompt Agents) optimises a DSPy program by using a separate **reflection LM** to analyse failures, propose improved instructions, and evolve the prompt across a Pareto front of candidates. The reflection LM is typically the strongest model available since it must reason about *why* outputs fail; here we point it at the top entry in `REFLECTION_MODEL_CANDIDATES`.

In [6]:
from typing import Literal, List

class NPSTopicClassifier(dspy.Signature):
    """Classify Net Promoter Score topics"""

    comment: str = dspy.InputField()
    answer: List[Literal['Slow or Unreliable Shipping', 'Inaccurate Product Descriptions or Photos', 'Limited Size or Shade Availability', 
                    'Unresponsive or Generic Customer Support', 'Website or App Bugs', 'Confusing Loyalty or Discount Systems', 
                    'Complicated Returns or Exchanges', 'Customs and Import Charges', 'Difficult Product Discovery', 
                    'Damaged or Incorrect Items']] = dspy.OutputField()

In [7]:
print(nps_data[0])
print(nps_data[0]['topics'])
print(wrap_text(nps_data[0]['comment'], width=72))

{'topics': ['Limited Size or Shade Availability'], 'comment': "Absolutely frustrated! Every time I find something I love, it's sold out in my size. What's the point of having a wishlist if nothing is ever available?"}
['Limited Size or Shade Availability']
Absolutely frustrated! Every time I find something I love, it's sold out
in my size. What's the point of having a wishlist if nothing is ever
available?


In [8]:
nps_topic_predictor_w_reasoning = dspy.ChainOfThought(NPSTopicClassifier)  # dspy CoT add a reasoning field to the prediction
response = nps_topic_predictor_w_reasoning(
    comment = "Absolutely frustrated! Every time I find something I love, it's sold out in my size." 
    "What's the point of having a wishlist if nothing is ever available?")
print(response)
print(response.reasoning)
print(response.answer)


Prediction(
    reasoning='The user expresses frustration because items they are interested in are consistently unavailable in their size. This directly relates to the "Limited Size or Shade Availability" category.',
    answer=['Limited Size or Shade Availability']
)
The user expresses frustration because items they are interested in are consistently unavailable in their size. This directly relates to the "Limited Size or Shade Availability" category.
['Limited Size or Shade Availability']


In [9]:
dspy.inspect_history(n = 1)






[2026-05-14T21:48:44.177044]

System message:

Your input fields are:
1. `comment` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (list[Literal['Slow or Unreliable Shipping', 'Inaccurate Product Descriptions or Photos', 'Limited Size or Shade Availability', 'Unresponsive or Generic Customer Support', 'Website or App Bugs', 'Confusing Loyalty or Discount Systems', 'Complicated Returns or Exchanges', 'Customs and Import Charges', 'Difficult Product Discovery', 'Damaged or Incorrect Items']]):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## comment ## ]]
{comment}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string", "enum": ["Slow or Unreliable Shipping", "Inaccurate Product Descriptions or Photos", "Limited Size or Shade Availability", "Unresponsive or Generic Customer Support", "Websit

In [10]:
nps_df = pd.DataFrame(nps_data)
# Net effect: each row gets a 1-based id (1, 2, 3, …, N) in a new column.
nps_df['id'] = list(map(lambda x: x + 1, range(nps_df.shape[0])))

In [11]:
tmp = []

for rec in tqdm.tqdm(nps_df.to_dict('records')):
    response = nps_topic_predictor_w_reasoning(comment = rec['comment'])  # dspy.InputField comment
    res = {
        'id': rec['id'],
        'predicted_topics': response.answer
    }

    tmp.append(res)

100%|██████████| 105/105 [01:38<00:00,  1.07it/s]


In [12]:
ini_model_topics_df = pd.DataFrame(tmp)

In [13]:
ini_model_topics_df

,id,predicted_topics
0,1,[Limited Size or Shade Availability]
1,2,[Difficult Product Discovery]
2,3,[Inaccurate Product Descriptions or Photos]
3,4,"[Confusing Loyalty or Discount Systems, Unresp..."
4,5,[Website or App Bugs]
...,...,...
100,101,"[Customs and Import Charges, Inaccurate Produc..."
101,102,"[Confusing Loyalty or Discount Systems, Compli..."
102,103,"[Limited Size or Shade Availability, Unrespons..."
103,104,"[Slow or Unreliable Shipping, Inaccurate Produ..."


In [14]:
nps_df = nps_df.merge(ini_model_topics_df)

In [15]:
# show head
nps_df.sample(5).to_dict('records')

[{'topics': ['Confusing Loyalty or Discount Systems'],
  'comment': 'Thought I qualified for free shipping with my membership but got charged anyway. Your terms and conditions are confusing and misleading.',
  'id': 12,
  'predicted_topics': ['Confusing Loyalty or Discount Systems']},
 {'topics': ['Customs and Import Charges'],
  'comment': 'Order got held at customs for weeks because of incorrect paperwork from your end. Completely avoidable delay.',
  'id': 59,
  'predicted_topics': ['Customs and Import Charges',
   'Slow or Unreliable Shipping']},
 {'topics': ['Website or App Bugs',
   'Confusing Loyalty or Discount Systems',
   'Damaged or Incorrect Items'],
  'comment': "Loyalty points didn't apply due to site glitch, and when the wrong item arrived, I lost the points entirely. System is completely unreliable.",
  'id': 105,
  'predicted_topics': ['Website or App Bugs',
   'Confusing Loyalty or Discount Systems',
   'Damaged or Incorrect Items']},
 {'topics': ['Confusing Loyalty o

In [16]:
def compare_topics(l1, l2):
    l1_fmt = ', '.join(sorted(l1))
    l2_fmt = ', '.join(sorted(l2))
    if l1_fmt == l2_fmt: 
        return 1 
    return 0


nps_df['model_accuracy'] = list(map(
    compare_topics,
    nps_df.topics,
    nps_df.predicted_topics
))

In [17]:
round(100*nps_df.model_accuracy.mean(), 2)

np.float64(88.57)

In [18]:
import random
random.random()

0.4360100068957914

In [19]:
trainset = []
valset = []
for rec in nps_data: 
    if random.random() <= 0.5:
        trainset.append(
            dspy.Example(
                comment = rec['comment'],
                answer = rec['topics']
            ).with_inputs('comment')
        )
    else: 
        valset.append(
            dspy.Example(
                comment = rec['comment'],
                answer = rec['topics']
            ).with_inputs('comment')
        )

In [20]:
def list_exact_match(example, pred, trace=None, pred_name=None, pred_trace=None):
    """Score topic-list predictions, with feedback for GEPA's reflector.

    Returns a ``dspy.Prediction(score=..., feedback=...)``:
      * ``score`` is 1.0 if the predicted topic set equals the gold set, else 0.0.
        BootstrapFewShotWithRandomSearch and MIPROv2 read this field.
      * ``feedback`` is a one-sentence diagnosis of the failure mode (missing
        topics, hallucinated topics, type mismatch). GEPA's ``reflection_lm``
        consumes this to rewrite the prompt; the other optimisers ignore it.
    """
    try:
        pred_answer = pred.answer
        expected_answer = example.answer

        if isinstance(pred_answer, list) and isinstance(expected_answer, list):
            pred_set = set(pred_answer)
            gold_set = set(expected_answer)
            score = 1.0 if pred_set == gold_set else 0.0
            if score == 1.0:
                feedback = "Correct: predicted topic set matches the gold set."
            else:
                missing = gold_set - pred_set
                extra = pred_set - gold_set
                parts = []
                if missing:
                    parts.append(f"missing topics: {sorted(missing)}")
                if extra:
                    parts.append(f"hallucinated topics not in gold: {sorted(extra)}")
                feedback = (
                    f"Incorrect. Gold: {sorted(gold_set)}. Predicted: {sorted(pred_set)}. "
                    + "; ".join(parts) + "."
                )
        else:
            score = 1.0 if pred_answer == expected_answer else 0.0
            verdict = "Correct" if score else "Incorrect"
            feedback = (
                f"{verdict}: predicted {pred_answer!r}, expected {expected_answer!r}."
            )
        return dspy.Prediction(score=score, feedback=feedback)
    except Exception as e:
        return dspy.Prediction(score=0.0, feedback=f"Metric error: {e}")

In [21]:
reflection_lm = dspy.LM(REFLECTION_MODEL_CANDIDATES[0], temperature=1.0, max_tokens=32768)

# GEPA metric must accept five arguments: (gold, pred, trace, pred_name, pred_trace)
gepa_tele_prompter = dspy.GEPA(
    metric=list_exact_match,
    auto="light",
    num_threads=24,
    reflection_lm=reflection_lm,
    track_stats=True,
)

In [22]:
gepa_optimized_nps_topic_predictor = gepa_tele_prompter.compile(
    nps_topic_predictor_w_reasoning,
    trainset=trainset,
    valset=valset,
)

2026/05/14 21:50:22 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 588 metric calls of the program. This amounts to 5.60 full evals on the train+val set.
2026/05/14 21:50:22 INFO dspy.teleprompt.gepa.gepa: Using 52 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|          | 0/588 [00:00<?, ?rollouts/s]2026/05/14 21:50:25 INFO dspy.evaluate.evaluate: Average Metric: 48.0 / 52 (92.3%)
2026/05/14 21:50:25 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.9230769230769231 over 52 / 52 examples
GEPA Optimization:   9%|▉         | 52/588 [00:03<00:32, 16.45rollouts/s]2026/05/14 21:50:25 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score

Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00,  3.25it/s]

2026/05/14 21:50:26 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/05/14 21:50:26 INFO dspy.teleprompt.gepa.gepa: Iteration 1: All subsample scores perfect. Skipping.
2026/05/14 21:50:26 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Reflective mutation did not propose a new candidate
GEPA Optimization:   9%|▉         | 55/588 [00:04<00:42, 12.52rollouts/s]2026/05/14 21:50:26 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Selected program 0 score: 0.9230769230769231



Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00,  3.31it/s]

2026/05/14 21:50:27 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/05/14 21:50:27 INFO dspy.teleprompt.gepa.gepa: Iteration 2: All subsample scores perfect. Skipping.
2026/05/14 21:50:27 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Reflective mutation did not propose a new candidate
GEPA Optimization:  10%|▉         | 58/588 [00:05<00:54,  9.81rollouts/s]2026/05/14 21:50:27 INFO dspy.teleprompt.gepa.gepa: Iteration 3: Selected program 0 score: 0.9230769230769231


...

2026/05/14 21:55:16 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 3 (66.7%)


2026/05/14 21:55:36 INFO dspy.teleprompt.gepa.gepa: Iteration 21: Proposed new text for predict: You are an expert AI assistant that analyzes and classifies customer feedback from Net Promoter Score (NPS) surveys. Your task is to accurately categorize a given customer comment into one or more predefined topics.

Your response must be structured as a JSON object with two keys: "reasoning" and "answer".
- The "reasoning" field should contain a brief, clear explanation of why you chose the specific topic(s). You should connect specific phrases or sentiments from the user's comment to your chosen categories.
- The "answer" field must be a list of strings, where each string is one of the predefined topics.

Follow these critical instructions to ensure accuracy:

1.  **Identify the Root Cause:** Carefully analyze the comment to determine the fundamental issue the user is describing. Do not classify based on secondary effects or symptoms. For example, if a comment says, "Order got held at cus

Average Metric: 2.00 / 3 (66.7%): 100%|██████████| 3/3 [00:00<00:00,  3.23it/s] 

2026/05/14 21:55:41 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 3 (66.7%)


2026/05/14 21:56:06 INFO dspy.teleprompt.gepa.gepa: Iteration 22: Proposed new text for predict: You are an AI assistant that classifies customer feedback from Net Promoter Score (NPS) surveys into a predefined list of topics.

Your goal is to accurately identify all the distinct **root causes** or **main subjects** of the customer's comment. It is crucial to distinguish between the primary, fundamental problem and any secondary consequences that arise from it.

### **Crucial Rules for Classification**

1.  **Focus Solely on the Root Cause:** Your primary goal is to identify the fundamental problem(s) the customer is describing. Always ask yourself: "What is the original, underlying issue that initiated this experience?"

2.  **CRITICAL: Distinguish Cause from Consequence.** This is the most important rule. A negative customer experience is often a chain reaction. You must only classify the *first link* in that chain (the root cause), not the subsequent effects.
    *   **Example of a 

In [23]:
gepa_optimized_nps_topic_predictor(
    comment = 
    "Absolutely frustrated! Every time I find something I love, it's sold out in my size. " 
    "What's the point of having a wishlist if nothing is ever available?"
)

Prediction(
    reasoning='The user is expressing frustration because items they want are consistently unavailable in their size. This directly points to an issue with the availability of specific sizes for the products offered.',
    answer=['Limited Size or Shade Availability']
)

In [24]:
dspy.inspect_history(n = 1)





[2026-05-14T21:56:12.048826]

System message:

Your input fields are:
1. `comment` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (list[Literal['Slow or Unreliable Shipping', 'Inaccurate Product Descriptions or Photos', 'Limited Size or Shade Availability', 'Unresponsive or Generic Customer Support', 'Website or App Bugs', 'Confusing Loyalty or Discount Systems', 'Complicated Returns or Exchanges', 'Customs and Import Charges', 'Difficult Product Discovery', 'Damaged or Incorrect Items']]):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## comment ## ]]
{comment}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string", "enum": ["Slow or Unreliable Shipping", "Inaccurate Product Descriptions or Photos", "Limited Size or Shade Availability", "Unresponsive or Generic Customer Support", "Websit

In [25]:
tmp = []

for e in tqdm.tqdm(valset):
    comment = e.comment 
    baseline_resp = nps_topic_predictor_w_reasoning(comment = comment) 
    gepa_resp = gepa_optimized_nps_topic_predictor(comment = comment)

    tmp.append(
        {
        'comment': comment,
        'gold_answer': e.answer,
        'baseline_answer': baseline_resp.answer,
        'gepa_answer': gepa_resp.answer
        }
    )

100%|██████████| 52/52 [01:33<00:00,  1.80s/it]


In [26]:
cmp_df = pd.DataFrame(tmp)

In [27]:
def list_exact_match_raw(expected_answer, pred_answer, trace=None):
    """Custom metric for comparing lists of topics"""
    try:
        # Convert to sets for order-independent comparison
        if isinstance(pred_answer, list) and isinstance(expected_answer, list):
            return set(pred_answer) == set(expected_answer)
        else:
            return pred_answer == expected_answer
    except Exception as e:
        print(f"Error in metric: {e}")
        return False

In [28]:
cmp_df['baseline_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.gold_answer,
    cmp_df.baseline_answer))

cmp_df['gepa_accuracy'] = list(map(
    list_exact_match_raw,
    cmp_df.gold_answer,
    cmp_df.gepa_answer))

In [29]:
cmp_df[['baseline_accuracy', 'gepa_accuracy']].mean()*100

baseline_accuracy    88.461538
gepa_accuracy        92.307692
dtype: float64

In [30]:
# --- 1. Predictions identical row-by-row, or just equally accurate?
def _key(x):
    """Order-independent comparable key for list[str] predictions."""
    return tuple(sorted(x)) if isinstance(x, list) else x
base_keys = cmp_df.baseline_answer.apply(_key)
gepa_keys = cmp_df.gepa_answer.apply(_key)
n_same   = (base_keys == gepa_keys).sum()
n_diff   = (base_keys != gepa_keys).sum()
print(f"Rows with IDENTICAL predictions: {n_same} / {len(cmp_df)}")
print(f"Rows where the two DIFFER:       {n_diff} / {len(cmp_df)}")
if n_diff:
    print("\nDisagreements:")
    print(cmp_df[base_keys != gepa_keys][
        ['gold_answer', 'baseline_answer', 'gepa_answer']
    ])
# --- 2. Did GEPA actually rewrite the instruction?
baseline_instr = nps_topic_predictor_w_reasoning.predict.signature.instructions
gepa_instr     = gepa_optimized_nps_topic_predictor.predict.signature.instructions
print(f"\nInstructions identical? {baseline_instr == gepa_instr}")
if baseline_instr != gepa_instr:
    print(f"Baseline instr ({len(baseline_instr)} chars): {baseline_instr[:200]}...")
    print(f"GEPA     instr ({len(gepa_instr)} chars): {gepa_instr[:200]}...")

Rows with IDENTICAL predictions: 49 / 52
Rows where the two DIFFER:       3 / 52

Disagreements:
                                          gold_answer  \
3                       [Difficult Product Discovery]   
32                              [Website or App Bugs]   
51  [Website or App Bugs, Confusing Loyalty or Dis...   

                                      baseline_answer  \
3   [Difficult Product Discovery, Website or App B...   
32  [Website or App Bugs, Unresponsive or Generic ...   
51  [Website or App Bugs, Damaged or Incorrect Items]   

                                          gepa_answer  
3   [Inaccurate Product Descriptions or Photos, Di...  
32                              [Website or App Bugs]  
51  [Website or App Bugs, Confusing Loyalty or Dis...  

Instructions identical? True
